<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/02_data_import/export_to_nifti_and_dicom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing and exporting data with pyCERR

## Introduction

pyCERR reads and writes the common medical-imaging formats. This notebook covers the import entry points, then exports scans, doses and structures to **NIfTI** and structures to a **DICOM RTSTRUCT**.

## Install pyCERR

In [ ]:
!pip install "pyCERR @ git+https://github.com/cerr/pyCERR.git"

## Import entry points

| Source | Function |
|---|---|
| DICOM directory | `pc.loadDcmDir(dcmDir)` |
| NIfTI scan | `pc.loadNiiScan(file, 'CT SCAN')` |
| NIfTI dose | `pc.loadNiiDose(file, scanNum, planC)` |
| NIfTI structure(s) | `pc.loadNiiStructure(file, scanNum, planC)` |
| pyCERR pickle | `pc.loadPlanCFromPkl(file)` |

Each appends to (or creates) a `PlanC`. Below we start from the sample DICOM CT and add a synthetic structure + dose to export.

## Set up a demo plan

In [ ]:
import numpy as np
import cerr.plan_container as pc
from cerr import datasets

# Real sample CT (downloaded & cached on first use)
dcmDir = datasets.fetch_sample_data('head_and_neck')
planC = pc.loadDcmDir(dcmDir)

# --- add two synthetic structures: a target (PTV) and an OAR (Parotid) ---
nR, nC, nS = (int(v) for v in planC.scan[0].getScanSize())
xV, yV, zV = planC.scan[0].getScanXYZVals()        # grid coords (cm)
Xc, Yc, Zc = np.meshgrid(xV, yV, zV, indexing='xy')
cx, cy, cz = xV[nC // 2], yV[nR // 2], zV[nS // 2]
distPTV2 = (Xc - cx) ** 2 + (Yc - cy) ** 2 + (Zc - cz) ** 2
ptvMask = distPTV2 < 3.0 ** 2                       # 3 cm sphere
oarMask = ((Xc - (cx + 4.0)) ** 2 + (Yc - cy) ** 2
           + (Zc - cz) ** 2) < 1.5 ** 2            # 1.5 cm sphere, offset
planC = pc.importStructureMask(ptvMask.astype(np.uint8), 0, 'PTV', planC)
planC = pc.importStructureMask(oarMask.astype(np.uint8), 0, 'Parotid', planC)

# --- add a synthetic dose: 70 Gy across the PTV with a Gaussian fall-off ---
PRESCRIPTION = 70.0
dist = np.sqrt(distPTV2)
dose3M = PRESCRIPTION * np.exp(-(np.clip(dist - 3.0, 0, None) ** 2)
                              / (2 * 2.5 ** 2))
dose3M[ptvMask] = PRESCRIPTION
planC = pc.importDoseArray(dose3M, xV, yV, zV, planC, 0,
                           {'fractionGroupID': 'Demo', 'units': 'GY'})
print('scans=%d  structures=%s  doses=%d'
      % (len(planC.scan), [s.structureName for s in planC.structure],
         len(planC.dose)))

## Export scans, doses & structures to NIfTI

`saveNii` writes a scan or dose; `saveNiiStructure` writes selected structures as a single label map (`dim=3`) or a 4-D stack (`dim=4`, for overlapping structures). We write to a temporary folder here.

In [ ]:
import os, tempfile
outDir = tempfile.mkdtemp()

planC.scan[0].saveNii(os.path.join(outDir, 'scan.nii.gz'))
planC.dose[0].saveNii(os.path.join(outDir, 'dose.nii.gz'))

labelDict = {planC.structure[i].structureName: i + 1 for i in (0, 1)}
pc.saveNiiStructure(os.path.join(outDir, 'structures.nii.gz'),
                    labelDict, planC, [0, 1])

print('wrote:', sorted(os.listdir(outDir)))

## Export structures to DICOM RTSTRUCT

`rtstruct_iod.create` writes the selected structures to a single RTSTRUCT file referencing the associated scan.

In [ ]:
from cerr.dcm_export import rtstruct_iod
rtFile = os.path.join(outDir, 'rtstruct.dcm')
rtstruct_iod.create([0, 1], rtFile, planC,
                    {'seriesDescription': 'Exported from pyCERR'})
print('wrote', os.path.basename(rtFile),
      '(%d bytes)' % os.path.getsize(rtFile))

## Notes

* The desktop GUI exposes these under **File -> Export** (NIfTI for scan/dose/structures, DICOM RTSTRUCT for structures).
* A whole `PlanC` can also be pickled - `import pickle; pickle.dump(planC, open(file, 'wb'))` - and reloaded with `pc.loadPlanCFromPkl(file)`.